# Dependents index statistics

This notebook computes the statistics of the dependents index, which is built from the Goblin snapshot of Maven Central (20 April 2025).

It shows the counts after each filtering step, the per library statistics for the whole index and for libraries with at least 50 popular dependents, and prints the LaTeX table used in the paper.

A popular dependent is a dependent project that at least one other project depends on (reach > 0).

**Needs** the index in `data/dependents-index` (see the pipeline README, `publish --download`) and `pip install -r requirements.txt`. The queries read about 136 million edges and take a few minutes.

In [ ]:
import os
from pathlib import Path

import duckdb

# The index folder. Set OSSBRIDGE_DATA to use another data folder.
DATA = Path(os.environ.get("OSSBRIDGE_DATA", Path.cwd().parent / "data"))
INDEX = DATA / "dependents-index"
MIN_POPULAR = 50

con = duckdb.connect()
con.execute("SET memory_limit = '10GB'")
print(INDEX)

## Filtering steps

0. **Raw Goblin graph.** Every dependency edge from a dependent release to a library version.
1. **Remove test-scope dependencies.** Test code is not in the published jar, so its calls cannot be analyzed.
2. **Remove inexact versions.** Ranges, properties, `LATEST`, `RELEASE` and `SNAPSHOT` do not tell which version is used. The pattern is the one in `DependentsIndex/build_index.py`.
3. **Remove duplicate edges.** The same release can declare the same library version twice with different scopes.
4. **Keep libraries with at least 50 popular dependents.**

In [ ]:
NOT_A_RELEASE = r"[][(),${}\s]|^(RELEASE|LATEST)$|SNAPSHOT"

con.execute(f"CREATE OR REPLACE VIEW step0 AS SELECT * FROM read_parquet('{INDEX / 'edges.parquet'}')")
con.execute("CREATE OR REPLACE TABLE step1 AS SELECT * FROM step0 WHERE coalesce(scope, '') <> 'test'")
con.execute(f"CREATE OR REPLACE TABLE step2 AS SELECT * FROM step1 WHERE NOT regexp_matches(library_version, '{NOT_A_RELEASE}')")
con.execute("""
    CREATE OR REPLACE TABLE step3 AS
    SELECT DISTINCT library_ga, library_version, dependent_ga, dependent_version FROM step2""")

# Reach of each project is the number of distinct projects that directly depend on it.
con.execute(f"""
    CREATE OR REPLACE TABLE reach AS
    SELECT ga, max(reach) AS reach FROM read_parquet('{INDEX / 'reach.parquet'}') GROUP BY ga""")

# One row per library after step 3, with its versions, dependents and popular dependents.
con.execute("""
    CREATE OR REPLACE TABLE libraries AS
    SELECT s.library_ga,
           count(DISTINCT s.library_version) AS versions,
           count(DISTINCT s.dependent_ga) AS dependents,
           count(DISTINCT s.dependent_ga) FILTER (WHERE r.reach > 0) AS popular
    FROM step3 s LEFT JOIN reach r ON r.ga = s.dependent_ga
    GROUP BY s.library_ga""")
con.execute(f"""
    CREATE OR REPLACE TABLE step4 AS
    SELECT s.* FROM step3 s JOIN libraries l USING (library_ga) WHERE l.popular >= {MIN_POPULAR}""")

In [ ]:
def counts(step, label):
    return f"""
        SELECT '{label}' AS step, count(*) AS edges,
               count(DISTINCT library_ga) AS libraries,
               count(DISTINCT (library_ga, library_version)) AS library_versions,
               count(DISTINCT dependent_ga) AS dependent_projects,
               count(DISTINCT (dependent_ga, dependent_version)) AS dependent_releases
        FROM {step}"""

funnel = con.sql(" UNION ALL ".join([
    counts("step0", "0. Raw Goblin graph"),
    counts("step1", "1. Remove test-scope dependencies"),
    counts("step2", "2. Remove inexact versions"),
    counts("step3", "3. Remove duplicate edges"),
    counts("step4", f"4. Keep libraries with >= {MIN_POPULAR} popular dependents"),
]) + " ORDER BY step")
funnel

## Popular dependent projects

In [ ]:
con.sql("""
    SELECT 'Whole index' AS scope, count(DISTINCT s.dependent_ga) AS popular_dependent_projects
    FROM step3 s JOIN reach r ON r.ga = s.dependent_ga WHERE r.reach > 0
    UNION ALL
    SELECT 'At least 50 popular dependents', count(DISTINCT s.dependent_ga)
    FROM step4 s JOIN reach r ON r.ga = s.dependent_ga WHERE r.reach > 0""")

## Per library statistics

Versions per library counts the versions that have at least one dependent after step 3.

In [ ]:
def per_library(where):
    return con.execute(f"""
        SELECT count(*), sum(versions),
               min(versions), avg(versions), median(versions), max(versions),
               min(popular), avg(popular), median(popular), max(popular),
               min(dependents), avg(dependents), median(dependents), max(dependents)
        FROM libraries {where}""").fetchone()

whole = per_library("")
kept = per_library(f"WHERE popular >= {MIN_POPULAR}")

names = ["libraries", "library versions",
         "versions min", "versions mean", "versions median", "versions max",
         "popular min", "popular mean", "popular median", "popular max",
         "dependents min", "dependents mean", "dependents median", "dependents max"]
for name, w, k in zip(names, whole, kept):
    print(f"{name:20} {w:>14,.2f} {k:>14,.2f}")

## Top 10 libraries by popular dependents

In [ ]:
con.sql(f"""
    SELECT library_ga, popular, dependents, versions FROM libraries
    WHERE popular >= {MIN_POPULAR} ORDER BY popular DESC, library_ga LIMIT 10""")

## LaTeX table

In [ ]:
def edges(step):
    return con.execute(f"SELECT count(*) FROM {step}").fetchone()[0]

def popular_projects(step):
    return con.execute(f"""
        SELECT count(DISTINCT s.dependent_ga) FROM {step} s
        JOIN reach r ON r.ga = s.dependent_ga WHERE r.reach > 0""").fetchone()[0]

def projects(step):
    return con.execute(f"SELECT count(DISTINCT dependent_ga) FROM {step}").fetchone()[0]

raw_edges = edges("step0")
n = lambda x: f"{x:,.0f}"
f = lambda x: f"{x:,.2f}"

rows = [
    ("Dependency edges", n(edges("step3")), n(edges("step4"))),
    ("Libraries", n(whole[0]), n(kept[0])),
    ("Library versions", n(whole[1]), n(kept[1])),
    ("Dependent projects", n(projects("step3")), n(projects("step4"))),
    ("Popular dependent projects", n(popular_projects("step3")), n(popular_projects("step4"))),
]
groups = [
    ("Versions per library", 2),
    ("Popular dependents per library", 6),
    ("All dependents per library", 10),
]

lines = [
    r"% Requires \usepackage{booktabs} and \usepackage{multirow}",
    r"\begin{table}[t]",
    r"\centering",
    r"\caption{Dependents index built from the Goblin snapshot of Maven Central (20 April 2025). "
    r"Popular dependents are dependent projects that at least one other project depends on (reach $>$ 0). "
    r"Test-scope dependencies, inexact versions and duplicate edges are removed. "
    f"The raw graph has {n(raw_edges)} edges before this filtering.}}",
    r"\label{tab:dependents-index}",
    r"\begin{tabular}{llrr}",
    r"\toprule",
    r" & & \textbf{Whole dump} & \textbf{$\geq$ " + str(MIN_POPULAR) + r" popular dependents} \\",
    r"\midrule",
]
lines += [rf"\multicolumn{{2}}{{l}}{{{label}}} & {w} & {k} \\" for label, w, k in rows]
for label, start in groups:
    lines.append(r"\midrule")
    lines.append(rf"\multirow{{4}}{{*}}{{{label}}}")
    for i, stat in enumerate(["Min", "Mean", "Median", "Max"]):
        fmt = f if stat == "Mean" else n
        lines.append(rf"  & {stat} & {fmt(whole[start + i])} & {fmt(kept[start + i])} \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
print("\n".join(lines))